# 06 — Public & Private Test Inference | DenseNet121

## 1. Môi trường và source code

In [7]:
import sys, subprocess, gc, json, zipfile
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from PIL import Image
from torch.utils.data import Dataset, DataLoader

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE)
REPO_DIR = Path('/kaggle/working/The-Impostor-AI')
REPO_URL = 'https://github.com/chienvoxn/The-Impostor-AI.git'
if not (REPO_DIR / '.git').exists():
    subprocess.run(['git','clone','--branch','master',REPO_URL,str(REPO_DIR)],check=True)
else:
    subprocess.run(['git','-C',str(REPO_DIR),'pull','--ff-only','origin','master'],check=True)
sys.path.insert(0,str(REPO_DIR/'src'))
from kemaodanh_oai.models import build_densenet121
from kemaodanh_oai.transforms import build_valid_transform
print('Source imported')

Device: cuda
Already up to date.
Source imported


From https://github.com/chienvoxn/The-Impostor-AI
 * branch            master     -> FETCH_HEAD


## 2. Tìm dữ liệu test và checkpoint

In [8]:
INPUT = Path('/kaggle/input')
WORK = Path('/kaggle/working')
OUT = WORK/'test_inference'
OUT.mkdir(parents=True,exist_ok=True)

def find_test_dir(name):
    matches = [p.parent for p in INPUT.rglob('pairs.csv') if p.parent.name == name and (p.parent/'images').is_dir()]
    if len(matches) != 1:
        raise FileNotFoundError(f'{name}: cần đúng 1 thư mục có pairs.csv và images/, tìm thấy: {matches}')
    return matches[0]

TEST_DIRS = {name: find_test_dir(name) for name in ('public_test','private_test')}
CHECKPOINTS = {}
for fold in range(3):
    matches = list(INPUT.rglob(f'densenet121_fold{fold}_best.pt'))
    if len(matches) != 1:
        raise FileNotFoundError(f'Checkpoint fold {fold}: cần đúng 1 file trong Kaggle Input, tìm thấy: {matches}')
    CHECKPOINTS[fold] = matches[0]
for name,path in TEST_DIRS.items(): print(name,':',path)
for fold,path in CHECKPOINTS.items(): print(f'Fold {fold}:',path)


public_test : /kaggle/input/datasets/chienvoo/dataset/the_imposter/the_imposter/data/public_test
private_test : /kaggle/input/datasets/chienvoo/dataset/the_imposter/the_imposter/data/private_test/private_test
Fold 0: /kaggle/input/datasets/chienvoo/densenet121-checkpoints/checkpoints/densenet121_fold0_best.pt
Fold 1: /kaggle/input/datasets/chienvoo/densenet121-checkpoints/checkpoints/densenet121_fold1_best.pt
Fold 2: /kaggle/input/datasets/chienvoo/densenet121-checkpoints/checkpoints/densenet121_fold2_best.pt


## 3. Kiểm tra schema và đường dẫn ảnh

In [9]:
test_pairs = {}
for name,folder in TEST_DIRS.items():
    df = pd.read_csv(folder/'pairs.csv',dtype={'pair_id':str})
    print(f'\n{name}: {len(df)} pairs; columns={list(df.columns)}')
    display(df.head(3))
    required = {'pair_id','image_0','image_1'}
    if not required.issubset(df.columns):
        raise ValueError(f'{name}: thiếu cột {required-set(df.columns)}')
    if df['pair_id'].isna().any() or not df['pair_id'].is_unique:
        raise ValueError(f'{name}: pair_id rỗng hoặc trùng')
    test_pairs[name] = df

def resolve_image(folder, rel):
    rel = Path(str(rel).replace('\\','/'))
    if rel.is_absolute() or '..' in rel.parts:
        raise ValueError(f'Đường dẫn ảnh không hợp lệ: {rel}')
    candidates = [folder/rel]
    if rel.parts and rel.parts[0] != 'images':
        candidates.append(folder/'images'/rel)
    found = [p for p in candidates if p.is_file()]
    if len(found) != 1:
        raise FileNotFoundError(f'Không tìm thấy duy nhất ảnh {rel} trong {folder}; thử: {candidates}')
    return found[0]

records=[]
for name,df in test_pairs.items():
    for row in df.itertuples(index=False):
        for pos in (0,1):
            records.append({'test_set':name,'pair_id':row.pair_id,'position':pos,
                            'image_path':str(resolve_image(TEST_DIRS[name],getattr(row,f'image_{pos}')))})
images_df = pd.DataFrame(records)
print('Images ready:',len(images_df), '| pairs:',sum(len(d) for d in test_pairs.values()))



public_test: 100 pairs; columns=['pair_id', 'image_0', 'image_1']


,pair_id,image_0,image_1
0,00001,images/00118.jpg,images/00198.jpg
1,00002,images/00136.jpg,images/00038.jpg
2,00003,images/00135.jpg,images/00129.jpg



private_test: 100 pairs; columns=['pair_id', 'image_0', 'image_1']


,pair_id,image_0,image_1
0,00001,images/00009.jpg,images/00101.jpg
1,00002,images/00082.jpg,images/00029.jpg
2,00003,images/00028.jpg,images/00110.jpg


Images ready: 400 | pairs: 200


## 4. Inference trên cả public và private test

In [10]:
class TestImages(Dataset):
    def __init__(self,frame):
        self.paths=frame['image_path'].tolist()
        self.transform=build_valid_transform(224,0.60)
    def __len__(self): return len(self.paths)
    def __getitem__(self,index):
        with Image.open(self.paths[index]) as im:
            return self.transform(im.convert('RGB'))

loader=DataLoader(TestImages(images_df),batch_size=16,shuffle=False,num_workers=2,
                  pin_memory=(DEVICE.type=='cuda'))
fold_probs={}
for fold,path in CHECKPOINTS.items():
    print(f'Fold {fold}: inference...',flush=True)
    model=build_densenet121(pretrained=False,num_classes=2).to(DEVICE)
    ckpt=torch.load(path,map_location='cpu',weights_only=True)
    if int(ckpt['fold'])!=fold: raise ValueError(f'Sai checkpoint fold {fold}')
    model.load_state_dict(ckpt['model_state_dict'],strict=True)
    model.eval()
    chunks=[]
    with torch.inference_mode():
        for batch in loader:
            batch=batch.to(DEVICE,non_blocking=True)
            with torch.autocast(device_type=DEVICE.type,enabled=(DEVICE.type=='cuda')):
                logits=model(batch)
            chunks.append(torch.softmax(logits.float(),dim=1)[:,1].cpu().numpy())
    probs=np.concatenate(chunks)
    assert len(probs)==len(images_df) and np.isfinite(probs).all()
    fold_probs[fold]=probs
    del model,ckpt
    gc.collect()
    if DEVICE.type=='cuda': torch.cuda.empty_cache()
    print(f'Fold {fold}: {len(probs)} images done')


Fold 0: inference...
Fold 0: 400 images done
Fold 1: inference...
Fold 1: 400 images done
Fold 2: inference...
Fold 2: 400 images done


## 5. Dự đoán theo cặp và lưu CSV

In [11]:
detail=images_df[['test_set','pair_id','position']].copy()
for fold in range(3): detail[f'prob_fake_fold{fold}']=fold_probs[fold]
detail['prob_fake_ensemble']=np.mean(np.stack([fold_probs[i] for i in range(3)]),axis=0)
detail_path=OUT/'densenet121_test_image_probabilities.csv'
detail.to_csv(detail_path,index=False)

saved=[detail_path]
for name,df in test_pairs.items():
    part=detail.loc[detail.test_set==name]
    wide=part.pivot(index='pair_id',columns='position',values='prob_fake_ensemble')
    assert wide.shape==(len(df),2) and wide.notna().all().all()
    wide=wide.rename(columns={0:'prob_fake_0',1:'prob_fake_1'})
    result=df[['pair_id']].merge(wide.reset_index(),on='pair_id',validate='one_to_one',how='left')
    assert result[['prob_fake_0','prob_fake_1']].notna().all().all()
    result['fake_position']=(result['prob_fake_1']>result['prob_fake_0']).astype(int)
    # Chỉ xuất pair_id và fake_position để tránh đưa cột xác suất vào submission.
    submission=result[['pair_id','fake_position']].copy()
    submission_path=OUT/f'densenet121_{name}_predictions.csv'
    submission.to_csv(submission_path,index=False)
    saved.append(submission_path)
    # File chi tiết riêng, phục vụ kiểm tra kết quả.
    details_path=OUT/f'densenet121_{name}_details.csv'
    result.to_csv(details_path,index=False)
    saved.append(details_path)
    print(f'\n{name}: {len(submission)} predictions')
    display(submission.head(5))
    print('Counts:',submission.fake_position.value_counts().sort_index().to_dict())

print('Lưu ý: pair_id,fake_position là định dạng dự đoán; hãy đối chiếu yêu cầu submission của cuộc thi trước khi nộp.')



public_test: 100 predictions


,pair_id,fake_position
0,00001,0
1,00002,0
2,00003,0
3,00004,1
4,00005,0


Counts: {0: 52, 1: 48}

private_test: 100 predictions


,pair_id,fake_position
0,00001,1
1,00002,1
2,00003,0
3,00004,1
4,00005,1


Counts: {0: 51, 1: 49}
Lưu ý: pair_id,fake_position là định dạng dự đoán; hãy đối chiếu yêu cầu submission của cuộc thi trước khi nộp.


## 6. Nén và tải kết quả về máy

In [12]:
zip_path=WORK/'densenet121_public_private_test_results.zip'
with zipfile.ZipFile(zip_path,'w',compression=zipfile.ZIP_DEFLATED) as z:
    for p in saved: z.write(p,arcname=p.name)
print('ZIP:',zip_path)
print('Included:',[p.name for p in saved])
print('Download ZIP từ Kaggle Output, lưu vào outputs/results/ trên máy.')


ZIP: /kaggle/working/densenet121_public_private_test_results.zip
Included: ['densenet121_test_image_probabilities.csv', 'densenet121_public_test_predictions.csv', 'densenet121_public_test_details.csv', 'densenet121_private_test_predictions.csv', 'densenet121_private_test_details.csv']
Download ZIP từ Kaggle Output, lưu vào outputs/results/ trên máy.
